In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [3]:
!pip install gensim

In [4]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [5]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [6]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [7]:
print("king")
print(glove_wiki_vectors.most_similar("king"))

king
[('prince', 0.7682329416275024), ('queen', 0.7507691383361816), ('son', 0.7020888328552246), ('brother', 0.6985775828361511), ('monarch', 0.6977890133857727), ('throne', 0.6919989585876465), ('kingdom', 0.6811409592628479), ('father', 0.680202841758728), ('emperor', 0.6712858080863953), ('ii', 0.6676074862480164)]


In [8]:
print("computer")
print(glove_wiki_vectors.most_similar("computer"))

computer
[('computers', 0.8751983046531677), ('software', 0.8373122215270996), ('technology', 0.7642159461975098), ('pc', 0.7366449236869812), ('hardware', 0.7290390729904175), ('internet', 0.72867751121521), ('desktop', 0.7234442830085754), ('electronic', 0.7221829295158386), ('systems', 0.7197922468185425), ('computing', 0.7141730785369873)]


In [9]:
print("money")
print(glove_wiki_vectors.most_similar("money"))

money
[('funds', 0.8508071303367615), ('cash', 0.8484836220741272), ('fund', 0.759483277797699), ('paying', 0.7415367364883423), ('pay', 0.740767240524292), ('paid', 0.7344814538955688), ('get', 0.733965277671814), ('tax', 0.7233582139015198), ('financing', 0.721599280834198), ('keep', 0.7185225486755371)]


In [10]:
print("love")
print(glove_wiki_vectors.most_similar("love"))

love
[('me', 0.73828125), ('passion', 0.735213577747345), ('my', 0.7327208518981934), ('life', 0.7287957668304443), ('dream', 0.7267670631408691), ('you', 0.7181724309921265), ('always', 0.7111519575119019), ('wonder', 0.7094581127166748), ('i', 0.7084633708000183), ('dreams', 0.7067317962646484)]


In [11]:
...

Ellipsis

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [12]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [13]:
for w1, w2 in word_pairs:
    similarity = glove_wiki_vectors.similarity(w1, w2)
    print(f"{w1} - {w2}: {similarity}")

coast - shore: 0.7000270485877991
clothes - closet: 0.5462760329246521
old - new: 0.6432487368583679
smart - intelligent: 0.7552731037139893
dog - cat: 0.8798074126243591
tree - lawyer: 0.07671947777271271


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [14]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [15]:
for word in test_words:
    print(f"{word}: {word in glove_wiki_vectors}")

covididiot: False
fomo: False
frenemies: True
anthropause: False
photobomb: False
selfie: False
pxg: False
pacg: False
cct: True
escc: True


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [16]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [17]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [18]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283244)

In [19]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.3518238)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [20]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [21]:
analogy("man", "engineer", "woman")

man : engineer :: woman : ?


,Analogy word,Score
0,technician,0.662020
1,educator,0.611515
2,engineers,0.592235
3,surgeon,0.588020
4,contractor,0.587330
5,engineering,0.568600
6,nurse,0.556342
7,chemist,0.554719
8,pioneer,0.546008
9,physician,0.545960


In [22]:
analogy("man", "programmer", "woman")

man : programmer :: woman : ?


,Analogy word,Score
0,educator,0.585333
1,programmers,0.573053
2,linguist,0.543201
3,technician,0.537336
4,freelance,0.536212
5,animator,0.533937
6,translator,0.533419
7,software,0.496662
8,psychotherapist,0.495662
9,technologist,0.488051


In [23]:
analogy("white", "wealth", "black")


white : wealth :: black : ?


,Analogy word,Score
0,riches,0.687385
1,vast,0.644604
2,fortune,0.620507
3,immense,0.606325
4,enormous,0.582452
5,rich,0.566496
6,income,0.559710
7,influence,0.547227
8,resources,0.546866
9,millions,0.544831


The results show that word embeddings can capture societal biases present in the training data.

For example, in the analogy "man : doctor :: woman", the top result is "nurse", which reflects a common gender stereotype associating women with nursing roles and men with higher-status medical professions. 

Similarly, in "man : engineer :: woman", some results such as "technician" and "educator" suggest a shift away from engineering roles, indicating a weaker but still present gender bias. 

In "man : programmer :: woman", the results include occupations like "educator" and "linguist", again showing a tendency to associate women with non-technical roles.

However, the race-related analogy "white : wealth :: black" returns general wealth-related terms like "riches" and "income", which does not clearly demonstrate strong bias in this case.

Overall, these examples show that word embeddings can reflect patterns and biases from real-world text (such as Wikipedia), and therefore may encode and reproduce societal stereotypes.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

The biases observed in the embeddings are concerning because they can reinforce and amplify existing societal stereotypes. For example, associating women with roles like "nurse" instead of "doctor" can perpetuate gender inequality in professional settings.

These biases become especially problematic when embeddings are used in real-world applications. For instance, in hiring systems that use machine learning models, biased embeddings could lead to unfair recommendations, such as favoring male candidates for technical roles or leadership positions.

Similarly, in search engines or recommendation systems, biased embeddings could influence the type of content shown to users, reinforcing stereotypes over time.

Overall, while word embeddings are powerful tools, it is important to be aware of their biases and take steps to mitigate their impact in real-world applications.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [24]:
from sklearn.datasets import fetch_20newsgroups

In [25]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

,text,target,target_name
0,"You know, I was reading 18 U.S.C. 922 and some...",6,talk.politics.guns
1,\n\n\nIt's not a bad question: I don't have an...,1,comp.graphics
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
...,...,...,...
4558,Hi Everyone ::\n\nI am looking for some soft...,1,comp.graphics
4559,Archive-name: x-faq/part3\nLast-modified: 1993...,2,comp.windows.x
4560,"\nThat's nice, but it doesn't answer the quest...",6,talk.politics.guns
4561,"Hi,\n I just got myself a Gateway 4DX-33V ...",2,comp.windows.x


In [26]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [27]:
!pip install spacy

In [28]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 6.0 MB/s  0:00:05 eta 0:00:010:01:02
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [29]:
import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [30]:
def preprocess(text):
    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if not token.is_stop and not token.is_punct and token.is_alpha:
            tokens.append(token.lemma_.lower())
    
    return " ".join(tokens)

In [31]:
df["text_pp"] = df["text"].apply(preprocess)

In [32]:
df[["text", "text_pp"]].head()

,text,text_pp
0,"You know, I was reading 18 U.S.C. 922 and some...",know read sence wonder help provide paragraph ...
1,\n\n\nIt's not a bad question: I don't have an...,bad question ref list algorithm think bit hard...
2,"\nActuallay I don't, but on the other hand I d...",actuallay hand support idea have newsgroup asp...
3,"The following problem is really bugging me,\na...",follow problem bug appreciate help create wind...
4,\n\n This is the latest from UPI \n\n For...,late upi foreign ministry spokesman ferhat ata...


In [33]:
df.iloc[2:6]

,text,target,target_name,text_pp
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics,actuallay hand support idea have newsgroup asp...
3,"The following problem is really bugging me,\na...",2,comp.windows.x,follow problem bug appreciate help create wind...
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast,late upi foreign ministry spokesman ferhat ata...
5,"Hi,\n I'd like to subscribe to Leadership Ma...",5,soc.religion.christian,hi like subscribe leadership magazine wonder d...


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

I experimented with different values for the number of topics and selected 8 topics as a reasonable choice. This is because the dataset includes 8 distinct categories, so setting n_components = 8 allows the model to align well with the underlying structure of the data.

When using fewer topics, multiple categories were merged together, making the topics less interpretable. When using more topics, the model produced redundant or overly specific topics that were harder to interpret.

Therefore, 8 topics provides a good balance between interpretability and capturing meaningful structure in the dataset.

In [34]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_df=0.95, min_df=2)
X = vectorizer.fit_transform(df["text_pp"])

In [35]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=8, random_state=42)
lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",8
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [36]:
X.shape

(4563, 15569)

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

In [37]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:]]
    print(f"Topic {topic_idx}: {top_words}")

Topic 0: ['section', 'april', 'israel', 'law', 'israeli', 'jewish', 'gm', 'state', 'right', 'jews']
Topic 1: ['version', 'include', 'server', 'run', 'available', 'window', 'program', 'use', 'image', 'file']
Topic 2: ['armenia', 'genocide', 'greek', 'turks', 'government', 'people', 'turkey', 'armenians', 'armenian', 'turkish']
Topic 3: ['get', 'think', 'season', 'win', 'good', 'player', 'year', 'play', 'team', 'game']
Topic 4: ['think', 'see', 'tell', 'file', 'know', 'come', 'go', 'entry', 'people', 'say']
Topic 5: ['good', 'say', 'question', 'god', 'like', 'time', 'know', 'israel', 'people', 'think']
Topic 6: ['kill', 'fire', 'control', 'crime', 'law', 'people', 'firearm', 'weapon', 'file', 'gun']
Topic 7: ['like', 'faith', 'thing', 'think', 'say', 'jesus', 'people', 'know', 'believe', 'god']


The LDA model produced several interpretable topics that align with the dataset categories.

For example, Topic 1 contains words such as “window”, “program”, “file”, and “image”, which clearly correspond to a “computer/technology” topic. Topic 2 includes words like “armenian”, “turkish”, “people”, and “war”, which suggests a “politics/international conflict” topic.

Topic 3 contains words such as “game”, “team”, “player”, and “season”, which strongly represents a “sports” topic. Topic 7 includes words like “jesus”, “god”, “faith”, and “believe”, indicating a “religion” topic.

Additionally, Topic 6 includes words like “firearm”, “weapon”, “gun”, and “law”, which aligns with discussions around “guns/politics”.

Overall, the topics are meaningful and correspond well to the original categories in the dataset, although some topics contain generic words (e.g., “people”, “think”), which slightly reduces interpretability.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [38]:
doc_topic_dist = lda.transform(X)

for i in range(5):
    topic = doc_topic_dist[i].argmax()
    print(f"Document {i} is mainly associated with Topic {topic}")

Document 0 is mainly associated with Topic 6
Document 1 is mainly associated with Topic 1
Document 2 is mainly associated with Topic 7
Document 3 is mainly associated with Topic 1
Document 4 is mainly associated with Topic 2


The document-topic assignments show that each document is primarily associated with a dominant topic.

From the results, Document 0 is mainly associated with Topic 6, Document 1 with Topic 1, Document 2 with Topic 7, Document 3 with Topic 1, and Document 4 with Topic 2. This indicates that the LDA model is able to assign a clear main topic to each document.

These dominant topics align with the themes identified earlier, such as politics, religion, sports, and technology. The presence of a single dominant topic per document suggests that the model is effectively capturing the main theme of each document.

Overall, the results demonstrate that LDA provides a meaningful representation of how topics are distributed across documents, while still allowing for the possibility of multiple underlying themes.

<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. Content-based filtering recommends items to users based on the similarity between item features and the user’s past preferences. It builds a profile of the user using previously liked or interacted items and then suggests new items with similar attributes. For example, if a user watches many action movies, the system will recommend other action movies.

2. One negative consequence of recommender systems is the creation of filter bubbles, where users are repeatedly exposed to similar content and not diverse perspectives. This can limit exploration and reinforce existing biases. Another issue is privacy concerns, as recommender systems often rely on collecting large amounts of personal data, which may be misused or lead to security risks.

3. Transfer learning in natural language processing involves using a model that has been pre-trained on a large dataset and adapting it to a specific task. Instead of training a model from scratch, knowledge learned from the pre-trained model is transferred to improve performance on tasks such as sentiment analysis or text classification, often requiring less data and time.

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)